In [1]:
infrastructure_urls = {
    "Serviceable_Airports": "https://www.globalfirepower.com/major-serviceable-airports-by-country.php",
    "Labor_Force": "https://www.globalfirepower.com/labor-force-by-country.php",
    "Ports_and_Terminals": "https://www.globalfirepower.com/major-ports-and-terminals.php",
    "Merchant_Marine": "https://www.globalfirepower.com/merchant-marine-strength-by-country.php",
    "Railway_Coverage_km": "https://www.globalfirepower.com/railway-coverage.php",
    "Roadway_Coverage_km": "https://www.globalfirepower.com/roadway-coverage.php"
}

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to load: {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value = record.find("div", class_="valueContainer")

        if value:

            if long_name:
                country = long_name.get_text(strip=True)

            elif short_name:
                country = short_name.get_text(strip=True)

            else:
                continue

            value = value.get_text(strip=True)

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in infrastructure_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Serviceable_Airports...
Scraping Labor_Force...
Scraping Ports_and_Terminals...
Scraping Merchant_Marine...
Scraping Railway_Coverage_km...
Scraping Roadway_Coverage_km...


In [6]:
infrastructure_df = dfs[0]

for df in dfs[1:]:

    infrastructure_df = infrastructure_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [7]:
for col in infrastructure_df.columns[1:]:

    infrastructure_df[col] = (
        infrastructure_df[col]
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    infrastructure_df[col] = pd.to_numeric(
        infrastructure_df[col],
        errors="coerce"
    )

In [8]:
print(infrastructure_df.head())

       Country  Serviceable_Airports  Labor_Force  Ports_and_Terminals  \
0  Afghanistan                    68      9133000                    0   
1      Albania                     3      1370000                    3   
2      Algeria                    95     13294000                   17   
3       Angola                   107     15961000                   21   
4    Argentina                   764     22286000                   37   

   Merchant_Marine  Railway_Coverage_km  Roadway_Coverage_km  
0                0                  NaN                  NaN  
1               69                  NaN                  NaN  
2              119                  NaN                  NaN  
3               64                  NaN                  NaN  
4              201                  NaN                  NaN  


In [9]:
infrastructure_df.to_csv("infrastructure.csv", index=False)

print("Infrastructure dataset saved successfully!")

Infrastructure dataset saved successfully!


In [10]:
print("Shape:", infrastructure_df.shape)
print()

print(infrastructure_df.info())
print()

print(infrastructure_df.head(10))

Shape: (145, 7)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Country               145 non-null    object 
 1   Serviceable_Airports  145 non-null    int64  
 2   Labor_Force           145 non-null    int64  
 3   Ports_and_Terminals   145 non-null    int64  
 4   Merchant_Marine       145 non-null    int64  
 5   Railway_Coverage_km   0 non-null      float64
 6   Roadway_Coverage_km   0 non-null      float64
dtypes: float64(2), int64(4), object(1)
memory usage: 8.1+ KB
None

       Country  Serviceable_Airports  Labor_Force  Ports_and_Terminals  \
0  Afghanistan                    68      9133000                    0   
1      Albania                     3      1370000                    3   
2      Algeria                    95     13294000                   17   
3       Angola                   107     15961000       